In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Loading Trained Models and Running Inference
# 
# This notebook shows you how to load your trained models and run predictions on new data.

# ## 1. Import Required Libraries

import torch
import torch.nn as nn
from PIL import Image
from transformers import (
    BertTokenizer, BertModel,
    ViTImageProcessor, ViTModel,
    CLIPProcessor, CLIPModel
)
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# ## 4. Load Multimodal (BERT+ViT) Model and Run Inference

# ### 4.1 Load Multimodal Model

def load_multimodal_model(checkpoint_path, device='cpu'):
    """Load trained multimodal BERT+ViT model"""
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Create model with same architecture
    model = MultimodalEngagementClassifier(
        bert_model_name=checkpoint['bert_model_name'],
        vit_model_name=checkpoint['vit_model_name'],
        num_classes=5,
        fusion_method=checkpoint['fusion_method']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    # Load tokenizer and processor
    tokenizer = BertTokenizer.from_pretrained(checkpoint['bert_model_name'])
    image_processor = ViTImageProcessor.from_pretrained(checkpoint['vit_model_name'])
    
    return model, tokenizer, image_processor, checkpoint['id_to_label']

# Load the model
multimodal_model, tokenizer, image_processor, mm_id_to_label = load_multimodal_model(
    'multimodal_engagement_classifier_complete.pth',
    device=device
)

print("✓ Multimodal model loaded successfully!")

# ### 4.2 Run Multimodal Inference

def predict_multimodal(image_path, title, description, model, tokenizer, 
                       image_processor, device, id_to_label, max_length=128):
    """Predict engagement using multimodal BERT+ViT model"""
    model.eval()
    
    # Process text
    text = f"{title} [SEP] {description}"
    text_inputs = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Process image
    image = Image.open(image_path).convert('RGB')
    image_inputs = image_processor(images=image, return_tensors='pt')
    
    # Move to device
    input_ids = text_inputs['input_ids'].to(device)
    attention_mask = text_inputs['attention_mask'].to(device)
    pixel_values = image_inputs['pixel_values'].to(device)
    
    # Predict
    with torch.no_grad():
        logits = model(input_ids, attention_mask, pixel_values)
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
    
    predicted_label = id_to_label[predicted_class]
    confidence = probabilities[0][predicted_class].item()
    
    # Get all probabilities
    all_probs = {id_to_label[i]: probabilities[0][i].item() 
                 for i in range(len(id_to_label))}
    
    return {
        'predicted_label': predicted_label,
        'confidence': confidence,
        'all_probabilities': all_probs
    }

# Example: Predict with image + text
image_path = "path/to/your/image.jpg"  # Update this
title = "Amazing content here!"
description = "This is a great post that everyone will love"

try:
    result = predict_multimodal(
        image_path, title, description,
        multimodal_model, tokenizer, image_processor, 
        device, mm_id_to_label
    )
    
    print(f"\n{'='*60}")
    print("MULTIMODAL MODEL PREDICTION")
    print(f"{'='*60}")
    print(f"Image: {image_path}")
    print(f"Title: {title}")
    print(f"Description: {description[:50]}...")
    print(f"\nPredicted Label: {result['predicted_label']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"\nAll Probabilities:")
    for label, prob in result['all_probabilities'].items():
        print(f"  {label:12s}: {prob:.2%}")
except FileNotFoundError:
    print(f"Image not found: {image_path}")

# ## 5. Batch Inference

# ### 5.1 Batch Inference with CLIP

def batch_predict_clip(image_paths, model, processor, device, id_to_label):
    """Predict multiple images at once"""
    model.eval()
    results = []
    
    for img_path in image_paths:
        try:
            result = predict_clip(img_path, model, processor, device, id_to_label)
            result['image_path'] = img_path
            results.append(result)
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
    
    return results

# Example: Batch prediction
image_paths = [
    "path/to/image1.jpg",
    "path/to/image2.jpg",
    "path/to/image3.jpg"
]

# Uncomment to run:
# batch_results = batch_predict_clip(image_paths, clip_model, clip_processor, device, clip_id_to_label)
# 
# print(f"\n{'='*60}")
# print("BATCH PREDICTIONS")
# print(f"{'='*60}")
# for result in batch_results:
#     print(f"\n{result['image_path']}: {result['predicted_label']} ({result['confidence']:.2%})")

# ### 5.2 Batch Inference with Multimodal Model from JSON

def batch_predict_from_json(json_path, model, tokenizer, image_processor, device, id_to_label):
    """Predict engagement for all samples in JSON dataset"""
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    results = []
    
    print(f"Processing {len(data)} samples...")
    for sample in data:
        try:
            result = predict_multimodal(
                sample['image'],
                sample['title'],
                sample['description'],
                model, tokenizer, image_processor,
                device, id_to_label
            )
            
            result['image_path'] = sample['image']
            result['title'] = sample['title']
            result['true_label'] = sample.get('engagement_label', 'Unknown')
            results.append(result)
            
        except Exception as e:
            print(f"Error processing {sample['image']}: {e}")
    
    return results

# Example: Process entire JSON dataset
# json_path = "your_test_dataset.json"
# 
# batch_results = batch_predict_from_json(
#     json_path, multimodal_model, tokenizer, 
#     image_processor, device, mm_id_to_label
# )
# 
# # Calculate accuracy
# correct = sum(1 for r in batch_results if r['predicted_label'] == r['true_label'])
# total = len(batch_results)
# accuracy = 100 * correct / total if total > 0 else 0
# 
# print(f"\n{'='*60}")
# print(f"BATCH RESULTS SUMMARY")
# print(f"{'='*60}")
# print(f"Total samples: {total}")
# print(f"Correct predictions: {correct}")
# print(f"Accuracy: {accuracy:.2f}%")

# ## 6. Save Predictions to File

def save_predictions_to_json(results, output_path):
    """Save predictions to JSON file"""
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Predictions saved to: {output_path}")

# Example:
# save_predictions_to_json(batch_results, "predictions_output.json")

# ## 7. Quick Inference Function (All-in-One)

def quick_predict(image_path, title=None, description=None, 
                  model_type='multimodal', checkpoint_path=None):
    """
    Quick prediction function for easy use
    
    Args:
        image_path: Path to image
        title: Title text (for multimodal only)
        description: Description text (for multimodal only)
        model_type: 'clip' or 'multimodal'
        checkpoint_path: Path to model checkpoint
    """
    
    if model_type == 'clip':
        if checkpoint_path is None:
            checkpoint_path = 'clip_engagement_classifier_complete.pth'
        
        model, processor, id_to_label = load_clip_model(checkpoint_path, device)
        result = predict_clip(image_path, model, processor, device, id_to_label)
    
    elif model_type == 'multimodal':
        if checkpoint_path is None:
            checkpoint_path = 'multimodal_engagement_classifier_complete.pth'
        
        if title is None or description is None:
            raise ValueError("Title and description required for multimodal model")
        
        model, tokenizer, img_proc, id_to_label = load_multimodal_model(checkpoint_path, device)
        result = predict_multimodal(
            image_path, title, description,
            model, tokenizer, img_proc, device, id_to_label
        )
    
    else:
        raise ValueError("model_type must be 'clip' or 'multimodal'")
    
    return result

# Example usage:
# result = quick_predict(
#     image_path="my_image.jpg",
#     title="Cool post",
#     description="Check this out!",
#     model_type='multimodal'
# )
# print(f"Prediction: {result['predicted_label']} ({result['confidence']:.2%})")

print("\n" + "="*60)
print("INFERENCE GUIDE COMPLETE!")
print("="*60)
print("\nYou can now:")
print("1. Load CLIP model: load_clip_model()")
print("2. Load Multimodal model: load_multimodal_model()")
print("3. Single prediction: predict_clip() or predict_multimodal()")
print("4. Batch prediction: batch_predict_clip() or batch_predict_from_json()")
print("5. Quick prediction: quick_predict()")
print("\nRemember to update image paths before running!")